# Use Case 01 — E-commerce Demand Forecasting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/usecases/notebooks/01-demand-forecasting.ipynb)

**CareerAlign — GCP Professional Machine Learning Engineer Use Cases**

This notebook demonstrates an end-to-end demand forecasting pipeline using synthetic retail data.
We replicate the core techniques used in a GCP-native production system:

- Generate realistic synthetic retail sales data (2 years, daily, with seasonality + trend + promotions + holidays)
- Exploratory data analysis and time series decomposition
- ARIMA modeling (local equivalent of BigQuery ML ARIMA_PLUS)
- Feature engineering for ML-based forecasting
- LightGBM / Prophet-style forecasting
- Model evaluation (MAPE, RMSE, bias)
- Visualization of forecasts vs actuals

All code runs end-to-end on synthetic data — no GCP credentials required.

## 0. Setup & Installation

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn statsmodels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Plot styling
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.facecolor': '#0b0d14',
    'figure.facecolor': '#0b0d14',
    'axes.edgecolor': '#2e3554',
    'axes.labelcolor': '#8e94b5',
    'xtick.color': '#545a78',
    'ytick.color': '#545a78',
    'text.color': '#e6e8f4',
    'grid.color': '#1a1f34',
    'grid.alpha': 0.5,
    'font.size': 11,
})

# Color palette matching CareerAlign design system
COLORS = {
    'amber': '#f59e0b',
    'blue': '#3b82f6',
    'cyan': '#22d3ee',
    'green': '#22c55e',
    'violet': '#a855f7',
    'red': '#ef4444',
    'text': '#e6e8f4',
    'muted': '#545a78',
}

print('Setup complete.')

## 1. Generate Synthetic Retail Sales Data

We create 2 years of daily sales data for a single SKU that mimics real e-commerce patterns:
- **Trend**: gradual upward growth (2% monthly)
- **Weekly seasonality**: higher sales on weekends
- **Annual seasonality**: peaks in November/December (holiday season), dips in January/February
- **Promotions**: random promotional events with 2–5x demand lift
- **Holidays**: major US holidays with specific demand effects
- **Noise**: random variation around the signal

In [ ]:
# --- Generate 2 years of daily dates ---
start_date = datetime(2022, 1, 1)
end_date = datetime(2023, 12, 31)
dates = pd.date_range(start=start_date, end=end_date, freq='D')
n_days = len(dates)

print(f'Generating {n_days} days of synthetic sales data...')
print(f'Date range: {dates[0].date()} to {dates[-1].date()}')

# --- Base demand (units/day) ---
base_demand = 100

# --- Trend component: 2% monthly growth ---
days_from_start = np.arange(n_days)
trend = base_demand * (1 + 0.02/30) ** days_from_start

# --- Weekly seasonality ---
# 0=Monday ... 6=Sunday; higher on Fri/Sat/Sun
day_of_week = np.array([d.weekday() for d in dates])
weekly_effect = np.where(
    day_of_week == 4, 1.15,    # Friday
    np.where(
        day_of_week == 5, 1.30,  # Saturday
        np.where(
            day_of_week == 6, 1.20,  # Sunday
            np.where(
                day_of_week == 0, 0.85,  # Monday (post-weekend dip)
                1.0  # Tue/Wed/Thu
            )
        )
    )
)

# --- Annual seasonality (cosine wave peaking in Nov/Dec) ---
day_of_year = np.array([d.timetuple().tm_yday for d in dates])
# Peak around day 330 (late November)
annual_effect = 1 + 0.25 * np.cos(2 * np.pi * (day_of_year - 330) / 365)

# --- US Holidays ---
us_holidays = {
    # (month, day): (name, demand_multiplier)
    (1, 1): ('New Year', 0.4),
    (2, 14): ('Valentines', 1.5),
    (7, 4): ('July 4th', 0.6),
    (10, 31): ('Halloween', 1.3),
    (11, 24): ('Black Friday 2022', 3.5),
    (11, 25): ('Black Friday+1 2022', 2.8),
    (11, 23): ('Black Friday 2023', 3.5),
    (11, 24): ('Black Friday+1 2023', 2.8),
    (12, 25): ('Christmas', 0.3),
    (12, 24): ('Christmas Eve', 1.8),
    (12, 26): ('Boxing Day', 2.0),
}

holiday_effect = np.ones(n_days)
is_holiday = np.zeros(n_days, dtype=bool)
holiday_names = [''] * n_days

for i, d in enumerate(dates):
    key = (d.month, d.day)
    if key in us_holidays:
        name, mult = us_holidays[key]
        holiday_effect[i] = mult
        is_holiday[i] = True
        holiday_names[i] = name

# --- Promotions (random, ~10% of days) ---
is_promotion = np.random.random(n_days) < 0.10
# Don't overlap with holidays
is_promotion = is_promotion & ~is_holiday
promo_discount = np.where(is_promotion, np.random.uniform(0.10, 0.40, n_days), 0.0)
# Promotion lift: 1.5x to 4x depending on discount depth
promo_effect = np.where(is_promotion, 1.0 + promo_discount * 8, 1.0)

# --- Combine all components ---
signal = trend * weekly_effect * annual_effect * holiday_effect * promo_effect

# --- Add noise (Poisson-like for count data) ---
noise_scale = 0.12
noise = np.random.normal(0, noise_scale * signal)
units_sold = np.maximum(0, np.round(signal + noise)).astype(int)

# --- Simulate weather (temperature) ---
# Sinusoidal with peak in July, trough in January
temp_max_f = 65 + 25 * np.sin(2 * np.pi * (day_of_year - 80) / 365) + np.random.normal(0, 5, n_days)

# --- Build DataFrame ---
df = pd.DataFrame({
    'date': dates,
    'units_sold': units_sold,
    'day_of_week': day_of_week,
    'month': [d.month for d in dates],
    'is_weekend': day_of_week >= 5,
    'is_holiday': is_holiday,
    'holiday_name': holiday_names,
    'is_promotion': is_promotion,
    'promo_discount': np.round(promo_discount, 2),
    'temp_max_f': np.round(temp_max_f, 1),
})

print(f'\nDataFrame shape: {df.shape}')
print(f'Total units sold: {df["units_sold"].sum():,}')
print(f'Promotion days: {df["is_promotion"].sum()} ({df["is_promotion"].mean()*100:.1f}%)')
print(f'Holiday days: {df["is_holiday"].sum()}')
df.head(10)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'hspace': 0.35})

# --- Plot 1: Full time series ---
ax = axes[0]
ax.plot(df['date'], df['units_sold'], color=COLORS['cyan'], alpha=0.6, linewidth=0.8, label='Daily Sales')
# Rolling 7-day average
rolling_7d = df['units_sold'].rolling(7).mean()
ax.plot(df['date'], rolling_7d, color=COLORS['amber'], linewidth=1.5, label='7-day Moving Avg')
# Mark promotions
promo_dates = df[df['is_promotion']]
ax.scatter(promo_dates['date'], promo_dates['units_sold'], 
           color=COLORS['green'], s=15, alpha=0.7, zorder=5, label='Promotions')
ax.set_title('Daily Unit Sales — Full History', fontweight='bold', fontsize=13)
ax.set_ylabel('Units Sold')
ax.legend(loc='upper left', framealpha=0.3)
ax.grid(True)

# --- Plot 2: Monthly aggregation ---
ax = axes[1]
monthly = df.set_index('date').resample('M')['units_sold'].agg(['sum', 'mean', 'std'])
ax.bar(monthly.index, monthly['sum'], width=20, color=COLORS['blue'], alpha=0.7, edgecolor=COLORS['blue'])
ax.set_title('Monthly Total Sales', fontweight='bold', fontsize=13)
ax.set_ylabel('Total Units')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.grid(True, axis='y')

# --- Plot 3: Day-of-week distribution ---
ax = axes[2]
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_means = df.groupby('day_of_week')['units_sold'].mean()
bars = ax.bar(dow_names, dow_means, color=COLORS['violet'], alpha=0.7, edgecolor=COLORS['violet'])
for bar, val in zip(bars, dow_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0f}', ha='center', va='bottom', color=COLORS['text'], fontsize=10)
ax.set_title('Average Sales by Day of Week', fontweight='bold', fontsize=13)
ax.set_ylabel('Avg Units Sold')
ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print('=== Sales Summary Statistics ===')
print(f'Mean daily sales:   {df["units_sold"].mean():.1f}')
print(f'Median daily sales: {df["units_sold"].median():.0f}')
print(f'Std deviation:      {df["units_sold"].std():.1f}')
print(f'Min:                {df["units_sold"].min()}')
print(f'Max:                {df["units_sold"].max()}')
print()
print('=== Promotion Impact ===')
promo_avg = df[df['is_promotion']]['units_sold'].mean()
non_promo_avg = df[~df['is_promotion'] & ~df['is_holiday']]['units_sold'].mean()
print(f'Avg sales (promo days):     {promo_avg:.1f}')
print(f'Avg sales (non-promo days): {non_promo_avg:.1f}')
print(f'Promotion lift:             {promo_avg/non_promo_avg:.2f}x')

## 3. Time Series Decomposition

Decompose the series into **trend**, **seasonal**, and **residual** components
using STL (Seasonal-Trend decomposition using LOESS).

In [ ]:
from statsmodels.tsa.seasonal import STL

# STL decomposition (period=7 for weekly seasonality)
ts = df.set_index('date')['units_sold'].asfreq('D').fillna(method='ffill')
stl = STL(ts, period=7, seasonal=13, trend=91, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 12), gridspec_kw={'hspace': 0.3})

components = [
    ('Observed', ts, COLORS['cyan']),
    ('Trend', result.trend, COLORS['amber']),
    ('Seasonal (Weekly)', result.seasonal, COLORS['green']),
    ('Residual', result.resid, COLORS['red']),
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data, color=color, linewidth=0.8 if title == 'Observed' else 1.2)
    ax.set_title(title, fontweight='bold', fontsize=12, loc='left')
    ax.grid(True)
    if title == 'Residual':
        ax.axhline(0, color=COLORS['muted'], linewidth=0.5, linestyle='--')

plt.suptitle('STL Decomposition (Weekly Period)', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f'Trend strength:    {1 - result.resid.var() / (result.trend + result.resid).var():.3f}')
print(f'Seasonal strength: {1 - result.resid.var() / (result.seasonal + result.resid).var():.3f}')

## 4. Train/Test Split

We use the last 30 days (December 2023) as the test set, and everything before as training.
This simulates a real production scenario where we forecast 2–4 weeks ahead.

In [ ]:
FORECAST_HORIZON = 30  # days

split_date = df['date'].max() - timedelta(days=FORECAST_HORIZON - 1)
train_df = df[df['date'] < split_date].copy()
test_df = df[df['date'] >= split_date].copy()

print(f'Training period: {train_df["date"].min().date()} to {train_df["date"].max().date()} ({len(train_df)} days)')
print(f'Test period:     {test_df["date"].min().date()} to {test_df["date"].max().date()} ({len(test_df)} days)')

## 5. ARIMA Model (Local Equivalent of BQML ARIMA_PLUS)

We use `statsmodels` SARIMAX as the local equivalent of BigQuery ML's ARIMA_PLUS.
In production on GCP, this would be a single SQL statement:

```sql
-- GCP: BigQuery ML ARIMA_PLUS (production version)
CREATE OR REPLACE MODEL demand_forecast.arima_demand_model
OPTIONS(
  model_type = 'ARIMA_PLUS',
  time_series_timestamp_col = 'sale_date',
  time_series_data_col = 'units_sold',
  auto_arima = TRUE,
  data_frequency = 'DAILY',
  holiday_region = 'US',
  clean_spikes_and_dips = TRUE
) AS
SELECT sale_date, units_sold
FROM demand_forecast.daily_sales;
```

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# Check stationarity with Augmented Dickey-Fuller test
train_ts = train_df.set_index('date')['units_sold'].asfreq('D').fillna(method='ffill')

adf_result = adfuller(train_ts, autolag='AIC')
print('=== Augmented Dickey-Fuller Test ===')
print(f'ADF Statistic: {adf_result[0]:.4f}')
print(f'p-value:       {adf_result[1]:.4f}')
print(f'Stationary:    {"Yes" if adf_result[1] < 0.05 else "No (needs differencing)"}')
print()

# Train SARIMAX(1,1,1)(1,0,1,7) — weekly seasonality
# This approximates BQML ARIMA_PLUS auto-selection
print('Training SARIMAX model (this may take a minute)...')
arima_model = SARIMAX(
    train_ts,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
arima_result = arima_model.fit(disp=False, maxiter=200)

print(f'AIC: {arima_result.aic:.1f}')
print(f'BIC: {arima_result.bic:.1f}')
print('\nModel Parameters:')
print(arima_result.params.round(4))

In [ ]:
# Generate ARIMA forecasts for the test period
arima_forecast = arima_result.get_forecast(steps=FORECAST_HORIZON)
arima_pred = arima_forecast.predicted_mean
arima_ci = arima_forecast.conf_int(alpha=0.05)

# Align indices
arima_pred.index = test_df['date'].values
arima_ci.index = test_df['date'].values

print(f'ARIMA forecast generated for {FORECAST_HORIZON} days.')
print(f'Forecast range: {arima_pred.min():.0f} to {arima_pred.max():.0f} units/day')

## 6. Feature Engineering for ML Approach

Create lag features, rolling statistics, and interaction features — the same features
we would build in BigQuery SQL for the Vertex AI custom model.

```sql
-- GCP: Feature engineering in BigQuery (production version)
SELECT
  sale_date, sku_id, units_sold,
  LAG(units_sold, 1) OVER (w) AS lag_1d,
  LAG(units_sold, 7) OVER (w) AS lag_7d,
  AVG(units_sold) OVER (w ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS avg_7d,
  STDDEV(units_sold) OVER (w ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS std_7d
FROM demand_forecast.daily_sales
WINDOW w AS (PARTITION BY sku_id ORDER BY sale_date)
```

In [ ]:
def create_features(dataframe):
    """Create lag, rolling, and calendar features for demand forecasting."""
    df_feat = dataframe.copy()
    
    # --- Lag features ---
    for lag in [1, 2, 3, 7, 14, 28]:
        df_feat[f'lag_{lag}d'] = df_feat['units_sold'].shift(lag)
    
    # --- Rolling window features ---
    for window in [7, 14, 28]:
        df_feat[f'rolling_mean_{window}d'] = df_feat['units_sold'].shift(1).rolling(window).mean()
        df_feat[f'rolling_std_{window}d'] = df_feat['units_sold'].shift(1).rolling(window).std()
    
    # --- Rolling min/max ---
    df_feat['rolling_min_7d'] = df_feat['units_sold'].shift(1).rolling(7).min()
    df_feat['rolling_max_7d'] = df_feat['units_sold'].shift(1).rolling(7).max()
    
    # --- Trend ratio (short/long moving average) ---
    df_feat['trend_ratio'] = df_feat['rolling_mean_7d'] / df_feat['rolling_mean_28d'].replace(0, np.nan)
    
    # --- Same day last week / last year ---
    df_feat['same_day_last_week'] = df_feat['units_sold'].shift(7)
    df_feat['same_day_last_year'] = df_feat['units_sold'].shift(364)
    
    # --- Calendar features ---
    df_feat['day_of_week'] = df_feat['date'].dt.dayofweek
    df_feat['day_of_month'] = df_feat['date'].dt.day
    df_feat['week_of_year'] = df_feat['date'].dt.isocalendar().week.astype(int)
    df_feat['month'] = df_feat['date'].dt.month
    df_feat['is_month_start'] = df_feat['date'].dt.is_month_start.astype(int)
    df_feat['is_month_end'] = df_feat['date'].dt.is_month_end.astype(int)
    
    # --- Cyclical encoding for day of week and month ---
    df_feat['dow_sin'] = np.sin(2 * np.pi * df_feat['day_of_week'] / 7)
    df_feat['dow_cos'] = np.cos(2 * np.pi * df_feat['day_of_week'] / 7)
    df_feat['month_sin'] = np.sin(2 * np.pi * df_feat['month'] / 12)
    df_feat['month_cos'] = np.cos(2 * np.pi * df_feat['month'] / 12)
    
    # --- Promotion interaction features ---
    df_feat['promo_weekend'] = (df_feat['is_promotion'] & df_feat['is_weekend']).astype(int)
    df_feat['promo_holiday'] = (df_feat['is_promotion'] & df_feat['is_holiday']).astype(int)
    
    # --- Days since last promotion ---
    promo_dates = df_feat[df_feat['is_promotion']].index
    days_since_promo = []
    last_promo_idx = -999
    for i in range(len(df_feat)):
        if df_feat.iloc[i]['is_promotion']:
            last_promo_idx = i
        days_since_promo.append(i - last_promo_idx if last_promo_idx >= 0 else 999)
    df_feat['days_since_promo'] = days_since_promo
    
    # --- Boolean to int ---
    df_feat['is_weekend'] = df_feat['is_weekend'].astype(int)
    df_feat['is_holiday'] = df_feat['is_holiday'].astype(int)
    df_feat['is_promotion'] = df_feat['is_promotion'].astype(int)
    
    return df_feat


df_features = create_features(df)

# Drop rows with NaN from lag/rolling features
df_features = df_features.dropna()
print(f'Feature matrix shape: {df_features.shape}')
print(f'Features created: {df_features.shape[1] - len(df.columns)}')
print(f'\nFeature columns:')
for col in sorted(df_features.columns):
    if col not in ['date', 'units_sold', 'holiday_name']:
        print(f'  {col}')

## 7. ML Model: Gradient Boosting (Equivalent of Vertex AI Custom Training)

We use scikit-learn's `GradientBoostingRegressor` as a local stand-in for LightGBM
trained on Vertex AI. In production:

```python
# GCP: Vertex AI custom training (production version)
from google.cloud import aiplatform

job = aiplatform.CustomTrainingJob(
    display_name='demand-forecast-lgbm',
    script_path='train.py',
    container_uri='us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0:latest',
    requirements=['lightgbm>=3.3'],
)
model = job.run(
    dataset=dataset,
    machine_type='n1-standard-8',
    args=['--n_estimators=500', '--learning_rate=0.05'],
)
```

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

# Define feature columns (exclude target and metadata)
exclude_cols = ['date', 'units_sold', 'holiday_name']
feature_cols = [c for c in df_features.columns if c not in exclude_cols]

# Split using the same date cutoff
train_feat = df_features[df_features['date'] < split_date]
test_feat = df_features[df_features['date'] >= split_date]

X_train = train_feat[feature_cols].values
y_train = train_feat['units_sold'].values
X_test = test_feat[feature_cols].values
y_test = test_feat['units_sold'].values

print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set:     {X_test.shape[0]} samples')

# Train Gradient Boosting model
print('\nTraining GradientBoostingRegressor...')
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_samples_leaf=10,
    subsample=0.8,
    random_state=42,
)
gb_model.fit(X_train, y_train)

# Predictions
gb_pred = gb_model.predict(X_test)
gb_pred = np.maximum(0, np.round(gb_pred))  # Non-negative integer predictions

print(f'Predictions generated.')
print(f'Predicted range: {gb_pred.min():.0f} to {gb_pred.max():.0f} units/day')

In [ ]:
# Feature importance analysis
importances = pd.Series(gb_model.feature_importances_, index=feature_cols)
top_features = importances.nlargest(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(top_features)), top_features.values, color=COLORS['amber'], alpha=0.8)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features.index)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Features — Gradient Boosting Model', fontweight='bold', fontsize=13)
ax.invert_yaxis()
ax.grid(True, axis='x')

for bar, val in zip(bars, top_features.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', color=COLORS['text'], fontsize=10)

plt.tight_layout()
plt.show()

## 8. Ensemble Model

Combine ARIMA and Gradient Boosting predictions with a weighted average.
This mirrors the production ensemble (40% ARIMA + 60% LightGBM) described in the use case.

```sql
-- GCP: Ensemble predictions in BigQuery (production version)
SELECT
  forecast_date, sku_id,
  ROUND(0.4 * arima_forecast + 0.6 * lgbm_forecast) AS ensemble_forecast
FROM arima_preds a
JOIN lgbm_preds l USING (forecast_date, sku_id)
```

In [ ]:
# Align ARIMA predictions with test set length
arima_aligned = arima_pred.values[:len(test_feat)]
gb_aligned = gb_pred[:len(test_feat)]
actual = test_feat['units_sold'].values
test_dates = test_feat['date'].values

# Weighted ensemble: 40% ARIMA + 60% Gradient Boosting
ARIMA_WEIGHT = 0.4
GB_WEIGHT = 0.6

ensemble_pred = np.round(ARIMA_WEIGHT * arima_aligned + GB_WEIGHT * gb_aligned)
ensemble_pred = np.maximum(0, ensemble_pred)

print(f'Ensemble predictions generated ({ARIMA_WEIGHT:.0%} ARIMA + {GB_WEIGHT:.0%} GB)')
print(f'Ensemble range: {ensemble_pred.min():.0f} to {ensemble_pred.max():.0f} units/day')

## 9. Model Evaluation

Evaluate all three models on MAPE, RMSE, and Bias — the key metrics for demand forecasting.

```sql
-- GCP: Evaluation in BigQuery (production version)
SELECT
  AVG(ABS(actual - forecast) / NULLIF(actual, 0)) * 100 AS mape,
  SQRT(AVG(POW(actual - forecast, 2))) AS rmse,
  AVG((forecast - actual) / NULLIF(actual, 0)) * 100 AS bias_pct
FROM demand_forecast.holdout_evaluation;
```

In [ ]:
def evaluate_forecast(actual, predicted, model_name='Model'):
    """Calculate MAPE, RMSE, MAE, and Bias for a forecast."""
    actual = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    
    # Filter out zeros in actual for MAPE calculation
    mask = actual > 0
    
    mape = np.mean(np.abs(actual[mask] - predicted[mask]) / actual[mask]) * 100
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    mae = np.mean(np.abs(actual - predicted))
    bias = np.mean((predicted[mask] - actual[mask]) / actual[mask]) * 100
    
    return {
        'Model': model_name,
        'MAPE (%)': round(mape, 1),
        'RMSE': round(rmse, 1),
        'MAE': round(mae, 1),
        'Bias (%)': round(bias, 1),
    }


# Also compute a naive baseline (last week's sales)
naive_pred = test_feat['same_day_last_week'].values

# Evaluate all models
results = pd.DataFrame([
    evaluate_forecast(actual, naive_pred, 'Naive (Last Week)'),
    evaluate_forecast(actual, arima_aligned, 'ARIMA (SARIMAX)'),
    evaluate_forecast(actual, gb_aligned, 'Gradient Boosting'),
    evaluate_forecast(actual, ensemble_pred, 'Ensemble (40/60)'),
])

print('=== Model Comparison ===')
print(results.to_string(index=False))
print()
best = results.loc[results['MAPE (%)'].idxmin()]
print(f'Best model: {best["Model"]} with MAPE = {best["MAPE (%)"]:.1f}%')

In [ ]:
# Evaluation bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics = ['MAPE (%)', 'RMSE', 'MAE']
colors_list = [COLORS['red'], COLORS['blue'], COLORS['amber'], COLORS['green']]

for ax, metric in zip(axes, metrics):
    values = results[metric].values
    bars = ax.bar(range(len(results)), values, color=colors_list, alpha=0.8)
    ax.set_xticks(range(len(results)))
    ax.set_xticklabels(['Naive', 'ARIMA', 'GB', 'Ensemble'], rotation=0)
    ax.set_title(metric, fontweight='bold', fontsize=12)
    ax.grid(True, axis='y')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', color=COLORS['text'], fontsize=10)

plt.suptitle('Model Evaluation Comparison', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Forecast Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'hspace': 0.3})

# --- Plot 1: Forecasts vs Actuals ---
ax = axes[0]

# Training tail (last 30 days for context)
context_days = 30
train_tail = train_df.tail(context_days)
ax.plot(train_tail['date'], train_tail['units_sold'],
        color=COLORS['muted'], linewidth=1.2, label='Training Data', alpha=0.6)

# Actuals
ax.plot(test_dates, actual, color=COLORS['text'], linewidth=2,
        label='Actual', marker='o', markersize=4)

# ARIMA
ax.plot(test_dates, arima_aligned, color=COLORS['blue'], linewidth=1.5,
        label=f'ARIMA (MAPE={results.iloc[1]["MAPE (%)"]:.1f}%)',
        linestyle='--', alpha=0.8)

# Gradient Boosting
ax.plot(test_dates, gb_aligned, color=COLORS['amber'], linewidth=1.5,
        label=f'GB (MAPE={results.iloc[2]["MAPE (%)"]:.1f}%)',
        linestyle='-.', alpha=0.8)

# Ensemble
ax.plot(test_dates, ensemble_pred, color=COLORS['green'], linewidth=2,
        label=f'Ensemble (MAPE={results.iloc[3]["MAPE (%)"]:.1f}%)')

# ARIMA confidence interval
ci_lower = arima_ci.iloc[:len(test_feat), 0].values
ci_upper = arima_ci.iloc[:len(test_feat), 1].values
ax.fill_between(test_dates, ci_lower, ci_upper, alpha=0.1, color=COLORS['blue'], label='95% CI (ARIMA)')

# Split line
ax.axvline(x=split_date, color=COLORS['red'], linestyle=':', alpha=0.5, label='Train/Test Split')

ax.set_title('Demand Forecast — All Models vs Actuals', fontweight='bold', fontsize=13)
ax.set_ylabel('Units Sold')
ax.legend(loc='upper left', fontsize=9, framealpha=0.3)
ax.grid(True)

# --- Plot 2: Forecast Errors ---
ax = axes[1]

arima_error = arima_aligned - actual
gb_error = gb_aligned - actual
ensemble_error = ensemble_pred - actual

ax.bar(np.arange(len(actual)) - 0.25, arima_error, width=0.25,
       color=COLORS['blue'], alpha=0.7, label='ARIMA Error')
ax.bar(np.arange(len(actual)), gb_error, width=0.25,
       color=COLORS['amber'], alpha=0.7, label='GB Error')
ax.bar(np.arange(len(actual)) + 0.25, ensemble_error, width=0.25,
       color=COLORS['green'], alpha=0.7, label='Ensemble Error')
ax.axhline(0, color=COLORS['muted'], linewidth=0.8, linestyle='--')
ax.set_title('Forecast Errors (Predicted - Actual)', fontweight='bold', fontsize=13)
ax.set_ylabel('Error (units)')
ax.set_xlabel('Day in Test Period')
ax.legend(loc='upper left', fontsize=9, framealpha=0.3)
ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 11. Forecast Accuracy by Horizon

In production, it's critical to understand how forecast accuracy degrades with
longer horizons. We measure MAPE for Day 1, Day 7, Day 14, and Day 30.

In [ ]:
horizons = [1, 3, 7, 14, 21, 30]
horizon_results = []

for h in horizons:
    if h > len(actual):
        break
    # Cumulative MAPE up to horizon h
    a = actual[:h]
    mask = a > 0
    if mask.sum() == 0:
        continue
    
    arima_mape_h = np.mean(np.abs(a[mask] - arima_aligned[:h][mask]) / a[mask]) * 100
    gb_mape_h = np.mean(np.abs(a[mask] - gb_aligned[:h][mask]) / a[mask]) * 100
    ens_mape_h = np.mean(np.abs(a[mask] - ensemble_pred[:h][mask]) / a[mask]) * 100
    
    horizon_results.append({
        'Horizon (days)': h,
        'ARIMA MAPE (%)': round(arima_mape_h, 1),
        'GB MAPE (%)': round(gb_mape_h, 1),
        'Ensemble MAPE (%)': round(ens_mape_h, 1),
    })

horizon_df = pd.DataFrame(horizon_results)
print('=== MAPE by Forecast Horizon ===')
print(horizon_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(horizon_df['Horizon (days)'], horizon_df['ARIMA MAPE (%)'],
        'o-', color=COLORS['blue'], linewidth=2, markersize=8, label='ARIMA')
ax.plot(horizon_df['Horizon (days)'], horizon_df['GB MAPE (%)'],
        's-', color=COLORS['amber'], linewidth=2, markersize=8, label='Gradient Boosting')
ax.plot(horizon_df['Horizon (days)'], horizon_df['Ensemble MAPE (%)'],
        'D-', color=COLORS['green'], linewidth=2, markersize=8, label='Ensemble')
ax.axhline(y=18, color=COLORS['red'], linestyle='--', alpha=0.5, label='Retraining Threshold (18%)')
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Forecast Accuracy Degradation by Horizon', fontweight='bold', fontsize=13)
ax.legend(framealpha=0.3)
ax.grid(True)
plt.tight_layout()
plt.show()

## 12. Production Deployment Notes

In a GCP production environment, this entire pipeline would be orchestrated as follows:

```python
# GCP: Cloud Composer DAG (production version)
from airflow import DAG
from airflow.providers.google.cloud.operators.bigquery import BigQueryInsertJobOperator
from airflow.providers.google.cloud.operators.vertex_ai.custom_job import CreateCustomTrainingJobOperator

with DAG('demand_forecast_daily', schedule_interval='0 4 * * *') as dag:
    # Step 1: Feature engineering in BigQuery
    feature_eng = BigQueryInsertJobOperator(
        task_id='feature_engineering',
        configuration={'query': {'query': FEATURE_SQL, 'useLegacySql': False}},
    )
    # Step 2: ARIMA_PLUS forecast
    arima_forecast = BigQueryInsertJobOperator(
        task_id='arima_forecast',
        configuration={'query': {'query': ARIMA_FORECAST_SQL, 'useLegacySql': False}},
    )
    # Step 3: Vertex AI batch prediction
    lgbm_batch = CreateCustomTrainingJobOperator(
        task_id='lgbm_batch_predict',
        ...
    )
    # Step 4: Ensemble & write to predictions table
    ensemble = BigQueryInsertJobOperator(
        task_id='ensemble_predictions',
        configuration={'query': {'query': ENSEMBLE_SQL, 'useLegacySql': False}},
    )
    feature_eng >> [arima_forecast, lgbm_batch] >> ensemble
```

### Key GCP Services Used:
| Service | Role |
|---------|------|
| BigQuery | Data warehouse, feature engineering, ARIMA_PLUS training |
| BigQuery ML | ARIMA_PLUS model training and forecasting |
| Vertex AI Training | Custom LightGBM model training |
| Vertex AI Model Registry | Model versioning and lineage |
| Vertex AI Model Monitoring | Data drift and prediction skew alerts |
| Cloud Composer | Pipeline orchestration (Airflow) |
| Cloud Scheduler | Daily trigger for batch predictions |
| Looker / Data Studio | Inventory planning dashboard |

In [ ]:
# Final summary
print('=' * 60)
print('  E-COMMERCE DEMAND FORECASTING — RESULTS SUMMARY')
print('=' * 60)
print()
print(f'  Dataset:          {n_days} days, single SKU')
print(f'  Training period:  {train_df["date"].min().date()} to {train_df["date"].max().date()}')
print(f'  Test period:      {test_df["date"].min().date()} to {test_df["date"].max().date()}')
print(f'  Forecast horizon: {FORECAST_HORIZON} days')
print()
print('  Model Performance:')
print(f'  {"Model":<25} {"MAPE":>8} {"RMSE":>8} {"Bias":>8}')
print(f'  {"-"*25} {"-"*8} {"-"*8} {"-"*8}')
for _, row in results.iterrows():
    print(f'  {row["Model"]:<25} {row["MAPE (%)"]!s:>7}% {row["RMSE"]!s:>8} {row["Bias (%)"]!s:>7}%')
print()
print(f'  Best Model: {best["Model"]} (MAPE = {best["MAPE (%)"]:.1f}%)')
print()
print('  Production Equivalent:')
print('    ARIMA         -> BigQuery ML ARIMA_PLUS')
print('    GradientBoost -> Vertex AI Custom Training (LightGBM)')
print('    Ensemble      -> BigQuery scheduled query')
print('    Monitoring    -> Vertex AI Model Monitoring')
print()
print('=' * 60)